# SFINCS vs HEC-RAS comparison — Manning roughness sensitivity

Case study: **Besaya River — Corrales de Buelna**  

Compares the response of two 2D hydraulic models to the same uncertainty
in Manning roughness coefficients:

| Model | Equations | Execution |
|--------|-----------|----------|
| **SFINCS** | Simplified inertial (no advective terms) | GPU Docker / SLURM |
| **HEC-RAS 6.6** | Full 2D Saint-Venant | COM (rascontrol, Windows) |

**Note**: This notebook requires notebooks 02 and 03 to have been run first
to produce `sfincs_sensitivity_results.csv` and `hecras_sensitivity_results.csv`,
or can load data directly from TIFFs via `load_flood_ensemble`.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from pyhydra.modeling.hydraulic.sensitivity import (
    load_flood_ensemble,
    compute_manning_stats,
    flooded_area,
    spatial_stats,
    filter_anomalous_simulations,
)

## Paths

In [2]:
import os
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'notebooks').exists() or (p / 'pyhydra').exists()),
    _cwd,
)
DATA_DIR  = Path(os.environ.get('HYDRA_DATA_DIR', str(REPO_ROOT / 'data')))
DATA_ROOT = DATA_DIR / 'pilot_cases' / 'manning_rugosidades' 
NB_DIR   = Path.cwd()
try:
    GEN_DIR  = DATA_ROOT / "generated"; GEN_DIR.mkdir(exist_ok=True)
except OSError:
    pass  # read-only filesystem in Docker

SFINCS_DIR       = DATA_ROOT / "sfincs_results"
HECRAS_DIR       = DATA_ROOT / "hecras_results" / "results"
COMBINATIONS_DIR = DATA_ROOT / "combinaciones" / "nsim_rugos"
MANNING_TIF      = DATA_ROOT / "manning.tif"

CELL_AREA_M2 = 25.0
THRESHOLD    = 0.05

_sfincs_ok = (GEN_DIR / "sfincs_sensitivity_results.csv").exists()
_hecras_ok = (GEN_DIR / "hecras_sensitivity_results.csv").exists()
_ext_ok    = _sfincs_ok and _hecras_ok
_has_comb = COMBINATIONS_DIR.exists() and any(COMBINATIONS_DIR.glob("combinacion_*.csv"))
_has_tif  = MANNING_TIF.exists()
_write_ok = True
try:
    (DATA_ROOT / ".write_test").touch()
    (DATA_ROOT / ".write_test").unlink()
except OSError:
    _write_ok = False
    print("⚠ Read-only filesystem — output files will not be saved")

## 1 · Load ensembles from both models

In [3]:
if _ext_ok:
    import re as _re
    _sfincs_nums = sorted([int(_re.search(r"(\d+)", p.stem).group(1))
                           for p in SFINCS_DIR.glob("hamax_sim_*.tif")])
    _hecras_nums = sorted([int(_re.search(r"(\d+)", p.stem).group(1))
                           for p in HECRAS_DIR.glob("hamax_sim_*.tif")])
    common_sims = np.intersect1d(_sfincs_nums, _hecras_nums)
    print(f"SFINCS files:   {len(_sfincs_nums)}")
    print(f"HEC-RAS files:  {len(_hecras_nums)}")
    print(f"Common sims:    {len(common_sims)}")


In [4]:
if _ext_ok and _has_comb and _has_tif:
    import gc

    # ── SFINCS ────────────────────────────────────────────────────────
    manning_stats_sf = reg_sf = calado_stats_sf = None
    try:
        flood_sf = load_flood_ensemble(str(SFINCS_DIR), threshold=THRESHOLD)
        flood_sf = flood_sf.sel(simulation=common_sims)
        print(f"SFINCS flood: {flood_sf.shape}")
        manning_stats_sf, reg_sf, calado_stats_sf = compute_manning_stats(
            raster_path=str(MANNING_TIF),
            combinations_dir=str(COMBINATIONS_DIR),
            flood_ensemble=flood_sf,
            simulation_numbers=common_sims.tolist(),
            cell_area_m2=CELL_AREA_M2,
            threshold=THRESHOLD,
        )
        print(f"SFINCS stats: {len(manning_stats_sf)} sims")
    except Exception as e:
        print(f"✗ SFINCS compute_manning_stats failed: {type(e).__name__}: {e}")
    finally:
        try: del flood_sf
        except NameError: pass
        gc.collect()

    # ── HEC-RAS ───────────────────────────────────────────────────────
    manning_stats_hr = reg_hr = calado_stats_hr = None
    try:
        flood_hr = load_flood_ensemble(str(HECRAS_DIR), threshold=THRESHOLD)
        flood_hr = flood_hr.sel(simulation=common_sims)
        print(f"HEC-RAS flood: {flood_hr.shape}")
        manning_stats_hr, reg_hr, calado_stats_hr = compute_manning_stats(
            raster_path=str(MANNING_TIF),
            combinations_dir=str(COMBINATIONS_DIR),
            flood_ensemble=flood_hr,
            simulation_numbers=common_sims.tolist(),
            cell_area_m2=CELL_AREA_M2,
            threshold=THRESHOLD,
        )
        print(f"HEC-RAS stats: {len(manning_stats_hr)} sims")
    except Exception as e:
        print(f"✗ HEC-RAS compute_manning_stats failed: {type(e).__name__}: {e}")
    finally:
        try: del flood_hr
        except NameError: pass
        gc.collect()

elif _ext_ok and not _has_tif:
    print(f"⚠ Manning raster not found: {MANNING_TIF}")
    manning_stats_sf = reg_sf = calado_stats_sf = manning_stats_hr = reg_hr = calado_stats_hr = None
elif _ext_ok:
    print("⚠ Combination CSVs not found — run notebook 01 first")
    manning_stats_sf = reg_sf = calado_stats_sf = manning_stats_hr = reg_hr = calado_stats_hr = None


## 2 · Per-model statistics

In [5]:
if _ext_ok and calado_stats_sf is not None:
    stats_sf = calado_stats_sf
    stats_hr = calado_stats_hr
    areas_sf = calado_stats_sf["flooded_area_m2"].values
    areas_hr = calado_stats_hr["flooded_area_m2"].values

    summary = pd.DataFrame({
        "SFINCS_calado_medio":   stats_sf["mean"].values,
        "HECRAS_calado_medio":   stats_hr["mean"].values,
        "SFINCS_area_km2":       areas_sf * 1e-6,
        "HECRAS_area_km2":       areas_hr * 1e-6,
    }, index=common_sims)

    summary


In [6]:
if _ext_ok and calado_stats_sf is not None:
    sf_filt_input = pd.DataFrame(
        {"depth_mean": stats_sf["mean"], "flooded_area_km2": areas_sf * 1e-6},
        index=common_sims,
    )
    hr_filt_input = pd.DataFrame(
        {"depth_mean": stats_hr["mean"], "flooded_area_km2": areas_hr * 1e-6},
        index=common_sims,
    )

    flagged, report = filter_anomalous_simulations(
        sf_filt_input, hr_filt_input,
        metrics=["depth_mean", "flooded_area_km2"],
        z_threshold=5.0,
    )

    print(f"Anomalous simulations detected: {flagged.sum()}")
    print(report.to_string())

    valid_sims = flagged[~flagged].index.values
    common_sims = valid_sims

    # Filter the already-computed stats DataFrames — no need to re-iterate the ensembles
    stats_sf = calado_stats_sf.loc[valid_sims]
    stats_hr = calado_stats_hr.loc[valid_sims]
    areas_sf = calado_stats_sf.loc[valid_sims, "flooded_area_m2"].values
    areas_hr = calado_stats_hr.loc[valid_sims, "flooded_area_m2"].values
    reg_sf   = reg_sf.loc[valid_sims]
    reg_hr   = reg_hr.loc[valid_sims]

    print(f"\nValid simulations for analysis: {len(common_sims)}")


## 2b · Detection and removal of anomalous simulations

A robust z-score (MAD-normalised) with threshold **k=5.0** is applied.

- **k=5.0** removes only evident numerical non-convergences (HEC-RAS area >1 km²,
  HEC-RAS depth >1.7 m). These are 5 simulations out of 1,000 (0.5%).
- **k=3.0** was discarded because the HEC-RAS area distribution is **bimodal**
  (two physically distinct inundation regimes); applying k=3 would eliminate both
  tails of that bimodal distribution, which is itself a scientifically relevant result.
- **IQR 1.5×** was also discarded for the same reason: it would remove ~11% of simulations.

## 3 · Compared distributions

In [7]:
if _ext_ok and calado_stats_sf is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    sns.kdeplot(stats_sf["mean"], ax=ax, label="SFINCS", fill=True, alpha=0.4)
    sns.kdeplot(stats_hr["mean"], ax=ax, label="HEC-RAS", fill=True, alpha=0.4)
    ax.set_xlabel("Mean spatial water depth (m)")
    ax.set_title("Mean water depth distribution — 1,000 sims")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    sns.kdeplot(areas_sf * 1e-6, ax=ax, label="SFINCS", fill=True, alpha=0.4)
    sns.kdeplot(areas_hr * 1e-6, ax=ax, label="HEC-RAS", fill=True, alpha=0.4)
    ax.set_xlabel("Flooded area (km²)")
    ax.set_title("Flooded area distribution — 1,000 sims")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.suptitle("SFINCS vs HEC-RAS — Besaya, T=500 years", y=1.01)
    plt.tight_layout()
    plt.show()

## 4 · Roughness ↔ depth regressions: both models

In [8]:
if _ext_ok and reg_sf is not None:
    print("Regression data already available: reg_sf and reg_hr")
    reg_sf.head()

In [9]:
if _ext_ok and reg_sf is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    for ax, reg, label, color in [
        (axes[0], reg_sf, "SFINCS",  "steelblue"),
        (axes[0], reg_hr, "HEC-RAS", "darkorange"),
    ]:
        valid = reg[["manning_mean", "depth_mean"]].dropna()
        coef = np.polyfit(valid["manning_mean"], valid["depth_mean"], 1)
        x_line = np.linspace(valid["manning_mean"].min(), valid["manning_mean"].max(), 100)
        ax.scatter(valid["manning_mean"], valid["depth_mean"],
                   alpha=0.3, s=8, color=color, label=label)
        ax.plot(x_line, np.polyval(coef, x_line), color=color, lw=2,
                label=f"{label}: y={coef[0]:.2f}x+{coef[1]:.2f}")

    axes[0].set_xlabel("Mean n Manning (wet cells)")
    axes[0].set_ylabel("Mean water depth (m)")
    axes[0].set_title("Roughness vs Water Depth")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    for ax, reg, label, color in [
        (axes[1], reg_sf, "SFINCS",  "steelblue"),
        (axes[1], reg_hr, "HEC-RAS", "darkorange"),
    ]:
        valid = reg[["manning_mean", "flooded_area_m2"]].dropna()
        coef = np.polyfit(valid["manning_mean"], valid["flooded_area_m2"] * 1e-6, 1)
        x_line = np.linspace(valid["manning_mean"].min(), valid["manning_mean"].max(), 100)
        axes[1].scatter(valid["manning_mean"], valid["flooded_area_m2"] * 1e-6,
                        alpha=0.3, s=8, color=color, label=label)
        axes[1].plot(x_line, np.polyval(coef, x_line), color=color, lw=2,
                     label=f"{label}: y={coef[0]:.2f}x+{coef[1]:.2f}")

    axes[1].set_xlabel("Mean n Manning (wet cells)")
    axes[1].set_ylabel("Flooded area (km²)")
    axes[1].set_title("Roughness vs Flooded Area")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.suptitle("SFINCS vs HEC-RAS — Manning sensitivity", y=1.01)
    plt.tight_layout()
    plt.show()

## 5 · Inter-model coherence: depth simulation-by-simulation

In [10]:
if _ext_ok and calado_stats_sf is not None:
    df_comp = pd.DataFrame({
        "sfincs_depth":  stats_sf["mean"],
        "hecras_depth":  stats_hr["mean"],
        "sfincs_area":   areas_sf * 1e-6,
        "hecras_area":   areas_hr * 1e-6,
    }, index=common_sims).dropna()

    r_depth, p_depth = stats.pearsonr(df_comp["sfincs_depth"], df_comp["hecras_depth"])
    r_area,  p_area  = stats.pearsonr(df_comp["sfincs_area"],  df_comp["hecras_area"])

    print(f"Depth correlation:  r={r_depth:.3f}  (p={p_depth:.2e})")
    print(f"Area correlation:   r={r_area:.3f}  (p={p_area:.2e})")

In [11]:
if _ext_ok and calado_stats_sf is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    ax.scatter(df_comp["sfincs_depth"], df_comp["hecras_depth"], alpha=0.35, s=10)
    lim = [df_comp[["sfincs_depth","hecras_depth"]].min().min() * 0.95,
           df_comp[["sfincs_depth","hecras_depth"]].max().max() * 1.05]
    ax.plot(lim, lim, "k--", lw=1, label="1:1")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("SFINCS — mean water depth (m)")
    ax.set_ylabel("HEC-RAS — mean water depth (m)")
    ax.set_title(f"Mean water depth sim-by-sim  (r={r_depth:.2f})")
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.scatter(df_comp["sfincs_area"], df_comp["hecras_area"], alpha=0.35, s=10)
    lim2 = [df_comp[["sfincs_area","hecras_area"]].min().min() * 0.95,
            df_comp[["sfincs_area","hecras_area"]].max().max() * 1.05]
    ax.plot(lim2, lim2, "k--", lw=1, label="1:1")
    ax.set_xlim(lim2); ax.set_ylim(lim2)
    ax.set_xlabel("SFINCS — flooded area (km²)")
    ax.set_ylabel("HEC-RAS — flooded area (km²)")
    ax.set_title(f"Flooded area sim-by-sim  (r={r_area:.2f})")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.suptitle("Inter-model coherence — same Manning combination", y=1.01)
    plt.tight_layout()
    plt.show()

## 6 · Coefficient of variation: which model is more sensitive to Manning?

In [12]:
if _ext_ok and calado_stats_sf is not None:
    def cv(series: pd.Series) -> float:
        """Coeficiente de variación en %."""
        return series.std() / series.mean() * 100

    metrics = pd.DataFrame({
        "Métrica":  ["Calado medio (m)", "Área inundada (km²)"],
        "SFINCS_mean":  [stats_sf["mean"].mean(), (areas_sf * 1e-6).mean()],
        "SFINCS_std":   [stats_sf["mean"].std(),  (areas_sf * 1e-6).std()],
        "SFINCS_CV%":   [cv(stats_sf["mean"]),    cv(pd.Series(areas_sf * 1e-6))],
        "HECRAS_mean":  [stats_hr["mean"].mean(), (areas_hr * 1e-6).mean()],
        "HECRAS_std":   [stats_hr["mean"].std(),  (areas_hr * 1e-6).std()],
        "HECRAS_CV%":   [cv(stats_hr["mean"]),    cv(pd.Series(areas_hr * 1e-6))],
    }).set_index("Métrica")

    metrics.round(3)

## 7 · Uncertainty maps (spatial standard deviation)

In [13]:
if _write_ok:
    if _ext_ok and calado_stats_sf is not None:
        out = pd.DataFrame({
            "sfincs_depth_mean":   stats_sf["mean"],
            "sfincs_depth_median": stats_sf["median"],
            "sfincs_area_km2":     areas_sf * 1e-6,
            "sfincs_manning_mean": reg_sf["manning_mean"],
            "hecras_depth_mean":   stats_hr["mean"],
            "hecras_depth_median": stats_hr["median"],
            "hecras_area_km2":     areas_hr * 1e-6,
            "hecras_manning_mean": reg_hr["manning_mean"],
        }, index=common_sims)

        out.to_csv(GEN_DIR / "comparison_sfincs_hecras_clean.csv")
        print(f"Guardado: {GEN_DIR}/comparison_sfincs_hecras_clean.csv  ({len(out)} filas)")

        report.to_csv(GEN_DIR / "anomalous_simulations.csv")
        print(f"Guardado: {GEN_DIR}/anomalous_simulations.csv  ({len(report)} filas)")

## 8 · Save combined results

In [14]:
if _write_ok:
    if _ext_ok and calado_stats_sf is not None:
        out = pd.DataFrame({
            "sfincs_depth_mean":   stats_sf["mean"],
            "sfincs_depth_median": stats_sf["median"],
            "sfincs_area_km2":     areas_sf * 1e-6,
            "sfincs_manning_mean": reg_sf["manning_mean"],
            "hecras_depth_mean":   stats_hr["mean"],
            "hecras_depth_median": stats_hr["median"],
            "hecras_area_km2":     areas_hr * 1e-6,
            "hecras_manning_mean": reg_hr["manning_mean"],
        }, index=common_sims)

        out.to_csv(GEN_DIR / "comparison_sfincs_hecras.csv")
        print(f"Guardado: {GEN_DIR}/comparison_sfincs_hecras.csv  ({len(out)} filas)")